In [ ]:
IPL Cricket Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, ttest_ind

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

matches = pd.read_csv("ipl_matches.csv")
players = pd.read_csv("ipl_player_stats.csv")

print("MATCHES shape:", matches.shape)
print("PLAYERS shape:", players.shape)
print(matches.head())

print("\n========== 1. TEAM PERFORMANCE ==========")

matches_played = pd.concat([matches["team1"], matches["team2"]]).value_counts()

wins = matches["winner"].value_counts()

team_summary = pd.DataFrame({
    "matches_played": matches_played,
    "wins": wins
}).fillna(0)
team_summary["losses"] = team_summary["matches_played"] - team_summary["wins"]
team_summary["win_pct"] = (team_summary["wins"] / team_summary["matches_played"] * 100).round(1)
team_summary = team_summary.sort_values("win_pct", ascending=False)

print(team_summary)

plt.figure()
plt.bar(team_summary.index, team_summary["win_pct"], color="#1f77b4")
plt.ylabel("Win %")
plt.title("Team Win Percentage")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("chart_1_team_win_pct.png")
plt.close()

top3_teams = team_summary.head(3).index.tolist()
season_wins = (
    matches[matches["winner"].isin(top3_teams)]
    .groupby(["season", "winner"])
    .size()
    .unstack(fill_value=0)
)

plt.figure()
for team in top3_teams:
    plt.plot(season_wins.index, season_wins[team], marker="o", label=team)
plt.xlabel("Season")
plt.ylabel("Wins")
plt.title("Season-wise Wins Trend (Top 3 Teams)")
plt.legend()
plt.tight_layout()
plt.savefig("chart_2_season_wins_trend.png")
plt.close()

print("\n========== 2. PLAYER STATISTICS ==========")

batting_summary = players.groupby("player").agg(
    total_runs=("runs_scored", "sum"),
    innings=("match_id", "count"),
    balls_faced=("balls_faced", "sum"),
    fours=("fours", "sum"),
    sixes=("sixes", "sum")
)
batting_summary["strike_rate"] = np.where(
    batting_summary["balls_faced"] > 0,
    (batting_summary["total_runs"] / batting_summary["balls_faced"]) * 100,
    0
).round(1)
batting_summary["avg_runs_per_innings"] = (
    batting_summary["total_runs"] / batting_summary["innings"]
).round(1)

top_run_scorers = batting_summary.sort_values("total_runs", ascending=False).head(10)
print("\nTop 10 Run Scorers:\n", top_run_scorers[["total_runs", "strike_rate"]])

bowling_summary = players.groupby("player").agg(
    total_wickets=("wickets_taken", "sum"),
    overs_bowled=("overs_bowled", "sum"),
    runs_conceded=("runs_conceded", "sum")
)
bowling_summary["economy"] = np.where(
    bowling_summary["overs_bowled"] > 0,
    bowling_summary["runs_conceded"] / bowling_summary["overs_bowled"],
    0
).round(2)

top_wicket_takers = bowling_summary.sort_values("total_wickets", ascending=False).head(10)
print("\nTop 10 Wicket Takers:\n", top_wicket_takers[["total_wickets", "economy"]])

plt.figure()
plt.barh(top_run_scorers.index[::-1], top_run_scorers["total_runs"][::-1], color="#ff7f0e")
plt.xlabel("Total Runs")
plt.title("Top 10 Run Scorers")
plt.tight_layout()
plt.savefig("chart_3_top_run_scorers.png")
plt.close()

plt.figure()
plt.barh(top_wicket_takers.index[::-1], top_wicket_takers["total_wickets"][::-1], color="#2ca02c")
plt.xlabel("Total Wickets")
plt.title("Top 10 Wicket Takers")
plt.tight_layout()
plt.savefig("chart_4_top_wicket_takers.png")
plt.close()

potm = matches["player_of_match"].value_counts().head(10)
print("\nTop 10 Player-of-the-Match Winners:\n", potm)

print("\n========== 3. MATCH-WINNING TRENDS ==========")

matches["toss_match_same"] = matches["toss_winner"] == matches["winner"]
toss_win_rate = matches["toss_match_same"].mean() * 100
print(f"\nTeams that won the toss also won the match: {toss_win_rate:.1f}% of the time")

contingency = pd.crosstab(matches["toss_decision"], matches["toss_match_same"])
chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f"Chi-square test (toss decision vs toss-winner-won-match): chi2={chi2:.2f}, p-value={p_value:.4f}")
if p_value < 0.05:
    print("-> Statistically significant relationship (p < 0.05)")
else:
    print("-> No statistically significant relationship (p >= 0.05)")

decision_outcome = matches.groupby("toss_decision")["toss_match_same"].mean() * 100
print("\nWin rate when toss winner chooses to BAT vs FIELD first:\n", decision_outcome.round(1))

plt.figure()
plt.bar(decision_outcome.index, decision_outcome.values, color=["#d62728", "#9467bd"])
plt.ylabel("Toss-winner win rate (%)")
plt.title("Bat First vs Field First — Win Rate")
plt.tight_layout()
plt.savefig("chart_5_toss_decision_winrate.png")
plt.close()

runs_margins = matches.loc[matches["win_by_runs"] > 0, "win_by_runs"]
wickets_margins = matches.loc[matches["win_by_wickets"] > 0, "win_by_wickets"]

print(f"\nAvg win margin (defending, by runs): {runs_margins.mean():.1f} runs")
print(f"Avg win margin (chasing, by wickets): {wickets_margins.mean():.1f} wickets")

plt.figure()
plt.hist(runs_margins, bins=15, color="#17becf", edgecolor="black")
plt.xlabel("Win Margin (Runs)")
plt.ylabel("Number of Matches")
plt.title("Distribution of Winning Margins (Defending Team)")
plt.tight_layout()
plt.savefig("chart_6_win_margin_runs.png")
plt.close()

venue_scores = matches.groupby("venue")["team1_score"].mean().sort_values(ascending=False).round(1)
print("\nAverage 1st innings score by venue:\n", venue_scores)

print("\nAll charts saved as PNG files in the current folder.")
print("Done.")